# 03장. TF-IDF — 글자를 숫자로

| 핵심 질문 | 학습 시간 |
|---|---:|
| AI는 메뉴 글자를 어떻게 숫자로 바꿀까? | 3회차 전반 · 약 90분 |


## 이 장에서 배울 내용

- 문자 n-gram이 무엇인지 예를 들어 설명할 수 있다.
- TF와 IDF가 각각 어떤 빈도를 보는지 말할 수 있다.
- 취향 문장과 메뉴를 숫자로 비교할 준비를 할 수 있다.


## 생각 열기

‘토마토파스타를 좋아한다’라는 취향과 ‘미트볼파스타’라는 메뉴는 비슷해 보입니다. 그러나 컴퓨터는 문장의 느낌을 그대로 비교하지 못합니다. 두 문장에 함께 나타나는 글자 조각을 세어 숫자로 표현해 봅시다.


## 핵심 용어

| 용어 | 뜻 |
|---|---|
| **n-gram** | 연속된 n개의 글자 조각 |
| **TF** | 한 문서 안에서 글자 조각이 나타난 정도 |
| **IDF** | 여러 문서 중 드물게 나타나 구별에 도움 되는 정도 |
| **벡터** | 여러 숫자를 순서대로 모은 표현 |


## 개념 익히기


반 학생 모두가 ‘급식’이라는 말을 쓰면 그 말만으로 누가 쓴 문장인지 구별하기 어렵습니다. 반대로 한 학생만 ‘파스타’라는 말을 썼다면 강한 단서가 됩니다.

TF는 한 메뉴 안에서 자주 나온 정도를, IDF는 여러 메뉴에 너무 흔한지 드문지를 봅니다. TF-IDF는 두 관점을 곱해 특징적인 글자 조각에 더 큰 값을 줍니다.

한국어 메뉴는 띄어쓰기가 일정하지 않을 수 있어 이 프로젝트는 단어 대신 2~4글자 문자 조각을 사용합니다.


## 활동 전 생각


‘파스타’를 2글자 조각으로 나누면 ‘파스’, ‘스타’가 됩니다.<br>
‘치즈파스타’에도 같은 조각이 있는지 찾아 동그라미를 쳐 보세요.

이제 A=`파스타 피자`, B=`파스타`, C=`밥 국` 세 문장을 생각합니다. `파스`는 3개 중 2개에 있지만 `피자`는 1개에만 있습니다. 따라서 `피자`의 IDF가 더 커서 A를 구별하는 특징이 됩니다. 이 장의 계산식은 `IDF = log((전체 문장 수+1)/(그 조각이 있는 문장 수+1))+1`입니다.


## 예상하기

- ‘파스타, 피자’를 좋아하는 가상 취향은 파스타·피자 메뉴와 가장 높은 유사도를 보인다.


## 활동 1. 문자 n-gram 관찰


### 코드 살펴보기


1. `_char_ngrams`는 문장을 2~4글자 조각과 반복 횟수로 바꿉니다.<br>
2. `example`은 관찰할 한 문장을 저장합니다.<br>
3. `_char_ngrams(example)`의 결과는 조각을 키, 반복 횟수를 값으로 가집니다.<br>
4. `list(grams.items())[:15]`는 조각이 너무 많지 않도록 앞의 15개만 보여 줍니다.


In [ ]:
import sys
from pathlib import Path

current_folder = Path.cwd().resolve()
for candidate in (current_folder, *current_folder.parents):
    if (candidate / "jupyter_course" / "notebook_support.py").is_file():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError(
        "프로젝트 폴더를 찾지 못했습니다. neis-meal-ai 폴더에서 "
        r".\.venv\Scripts\python.exe -m notebook 명령으로 다시 시작하세요."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from jupyter_course.notebook_support import course_setup

setup = course_setup(PROJECT_ROOT)
PROJECT_ROOT = setup["root"]
raw_rows = setup["rows"]
meal_df = setup["frame"]
data_source = setup["source"]
print("프로젝트 폴더:", PROJECT_ROOT)
print("데이터 출처:", data_source)
print("급식 행 수:", len(raw_rows))

from neis_meal_ai.recommender import _char_ngrams, _tfidf_similarity

example = "치즈 파스타"
grams = _char_ngrams(example)
print("2~4글자 조각 일부:")
print(list(grams.items())[:15])


### 결과 해석하기

공백 주변에도 표시를 더해 단어의 시작과 끝을 구별합니다. 같은 조각이 반복되면 빈도가 올라갑니다.


## 활동 2. 작은 TF-IDF를 직접 계산


### 코드 살펴보기


1. `tiny_documents`는 손으로 확인할 세 문장을 저장합니다.<br>
2. `_char_ngrams`는 각 문장의 조각별 횟수를 만들고, 다음 줄은 그 조각이 들어 있는 문장 수를 셉니다.<br>
3. `횟수 / 전체 조각 수`가 길이가 다른 문장을 공정하게 비교하는 TF입니다.<br>
4. `math.log(...) + 1`로 IDF를 구한 뒤 TF와 곱하면 한 문장의 TF-IDF가 됩니다.


In [ ]:
import math

tiny_documents = ["파스타 피자", "파스타", "밥 국"]
term = "피자"
tiny_counts = [_char_ngrams(text) for text in tiny_documents]
document_frequency = sum(term in counts for counts in tiny_counts)
idf = math.log((len(tiny_documents) + 1) / (document_frequency + 1)) + 1
tf = tiny_counts[0][term] / sum(tiny_counts[0].values())
tfidf = tf * idf

print("전체 문장 수:", len(tiny_documents))
print("'피자'가 있는 문장 수:", document_frequency)
print("TF:", round(tf, 3), "IDF:", round(idf, 3), "TF-IDF:", round(tfidf, 3))


### 결과 해석하기

‘피자’는 첫 문장의 전체 n-gram 중 한 번 있으므로 TF는 `1 ÷ 전체 조각 수`입니다. 세 문장 중 한 문장에만 있으므로 흔한 조각보다 큰 IDF를 얻습니다. 실제 추천기도 이 정규화 계산을 많은 글자 조각에 반복합니다.


## 활동 3. 가상 취향과 다섯 메뉴 비교


### 코드 살펴보기


1. `query`는 실제 학생 정보가 아닌 가상 취향 문장입니다.<br>
2. `tolist()`는 표의 메뉴 열을 문장 목록으로 바꿉니다.<br>
3. `_tfidf_similarity`는 취향과 각 메뉴의 방향 가까움을 한 번에 계산합니다.<br>
4. `zip(...)`은 같은 위치의 날짜·점수·메뉴를 한 줄씩 묶어 출력합니다.


In [ ]:
query = "파스타 피자 면"
menu_texts = meal_df["menu_text"].tolist()
similarity_array = _tfidf_similarity(menu_texts, query)
similarities = [round(float(value), 3) for value in similarity_array]

for date, score, menu in zip(meal_df["date"], similarities, menu_texts):
    print(date, score, menu[:45])

chapter_result = {
    "chapter": "03",
    "similarities": similarities,
    "query": query,
}


### 결과 해석하기

유사도는 0에 가까울수록 공통 글자 특징이 적고, 1에 가까울수록 방향이 비슷합니다. 점수는 만족도나 건강 점수가 아닙니다.


## 탐구 활동

query의 단어를 ‘밥 국물’ 또는 자신이 고른 가상 취향으로 바꾸고 가장 높은 날짜를 찾으세요.

먼저 기본값으로 실행한 뒤 한 곳만 바꾸어 결과를 비교합니다.


In [ ]:
practice_query = "밥 국물"
practice_scores = _tfidf_similarity(meal_df["menu_text"].tolist(), practice_query)
best_index = int(practice_scores.argmax())
print("가상 취향:", practice_query)
print("가장 비슷한 메뉴:", meal_df.iloc[best_index]["menu_text"])
print("유사도:", round(float(practice_scores[best_index]), 3))


### 관찰 기록

- 바꾼 것:  
- 달라진 결과:  
- 그렇게 된 까닭:


## 확인 문제

1. 문자 2-gram은 무엇인가요?
2. 모든 메뉴에 흔한 글자보다 일부 메뉴에만 있는 글자가 구별에 유리한 이유는 무엇인가요?
3. TF-IDF 유사도가 높으면 반드시 맛있거나 건강하다는 뜻인가요?


## 정답과 해설


1. 연속된 두 글자 조각입니다.<br>
2. 드문 글자가 메뉴의 특징을 더 잘 나타내기 때문입니다.<br>
3. 아닙니다. 입력한 취향 글자와 메뉴 글자의 특징이 비슷하다는 뜻뿐입니다.


## 핵심 정리

- 문자 n-gram은 문장을 짧은 글자 조각으로 나눈다.
- TF-IDF는 한 메뉴에서 중요하고 전체에서는 드문 특징을 크게 본다.
- 텍스트 유사도는 취향 표현의 가까움이지 정답이 아니다.

### 다음 장에서 배울 내용

04장에서는 벡터의 방향을 비교하고 숫자 특징이 비슷한 식단을 묶습니다.


In [ ]:
import json
print("__CHAPTER_RESULT__=" + json.dumps(chapter_result, ensure_ascii=False))
